This specific code will be executed as a once-off process and is intended to perform an update on the ProductDescription field. It is designed for a single-use execution and will not be part of the regular application flow.


In [0]:
%sql
MERGE INTO `p-cdh`.derivativeactive.dexposure AS t
USING (
    SELECT DISTINCT 
        ep.ExposureKey,
        ep.BusinessDate,
        qc_qca.SubProductDescription
    FROM `p-cdh`.primary.exposure ep
    INNER JOIN (
        SELECT DISTINCT SUBSTRING(ex.ExposureKey, 4, 50) AS joinkey, 
               p.ProductDescription AS SubProductDescription,
               ex.BusinessDate
        FROM `p-cdh`.primary.exposure ex
        INNER JOIN `p-cdh`.primary.product p 
            ON p.ProductKey = ex.ProductKey 
            AND p.SourceSystem = ex.SourceSystem 
            AND p.BusinessDate = ex.BusinessDate
        WHERE ex.SourceSystem = 'qc' 
          AND ex.PrimaryPartyFlag = true
        
        UNION
        
        SELECT DISTINCT ex.ExposureKey AS joinkey, 
               p.ProductDescription AS SubProductDescription,
               ex.BusinessDate
        FROM `p-cdh`.primary.exposure ex
        INNER JOIN `p-cdh`.primary.product p 
            ON p.ProductKey = ex.ProductKey 
            AND p.SourceSystem = ex.SourceSystem 
            AND p.BusinessDate = ex.BusinessDate
        WHERE ex.SourceSystem = 'qca' 
          AND ex.PartyRelationshipTypeCode = '5000'
    ) qc_qca 
        ON SUBSTRING(ep.ExposureKey, 4, 50) = qc_qca.joinkey
        AND ep.BusinessDate = qc_qca.BusinessDate
    WHERE ep.SourceSystem = 'profits' 
      AND ep.PartyRelationshipType = 'Primary Owner'
) AS s
ON t.ExposureKey = s.ExposureKey 
   AND t.BusinessDate = s.BusinessDate
WHEN MATCHED
THEN UPDATE SET t.SubProductDescription = s.SubProductDescription;
